In [2]:
from mmpose.apis.inference import inference_topdown
from mmpose.apis import init_model as init_pose_estimator
from mmpose.structures import merge_data_samples, split_instances
from mmpose.registry import VISUALIZERS

import cv2
import mmcv
import numpy as np
import os
import json_tricks as json
import time
import pickle

In [3]:
show = False
draw_bbox = True
kpt_thresh = 0.2
radius = 5
thickness = 3

resolution = (3840, 2160)

video_name = 'gopro1_synced'

video_file = f'inputs/{video_name}.mp4'
input_file = f'bboxes/tracked_bboxes_{video_name}.pkl'
output_file = f'vis_dir/updated_{video_name}.mp4'
save_path = f'pose_outputs/updated_{video_name}.json'

In [4]:
def process_one_image(img,
                      bboxes,
                      pose_estimator,
                      visualizer=None,
                      show_interval=0):
    """Visualize predicted keypoints of one image."""
    # predict keypoints
    pose_results = inference_topdown(pose_estimator, img, bboxes)
    data_samples = merge_data_samples(pose_results)

    # show the results
    if isinstance(img, str):
        img = mmcv.imread(img, channel_order='rgb')
    elif isinstance(img, np.ndarray):
        img = mmcv.bgr2rgb(img)

    if visualizer is not None:
        visualizer.add_datasample(
            'result',
            img,
            data_sample=data_samples,
            draw_gt=False,
            draw_heatmap=False,
            draw_bbox=draw_bbox,
            show_kpt_idx=False,
            skeleton_style='mmpose',
            show=show,
            wait_time=show_interval,
            kpt_thr=kpt_thresh)

    # if there is no instance detected, return None
    return data_samples.get('pred_instances', None)


In [5]:
# build pose estimator
pose_estimator = init_pose_estimator(
    'configs/hand_2d_keypoint/rtmpose/hand5/rtmpose-m_8xb256-210e_hand5-256x256.py',
    'checkpoints/hands/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth',
    device='cuda',
    cfg_options=dict(
        model=dict(test_cfg=dict(output_heatmaps=False))))

# build visualizer
pose_estimator.cfg.visualizer.radius = radius
pose_estimator.cfg.visualizer.alpha = 0.8
pose_estimator.cfg.visualizer.line_width = thickness
visualizer = VISUALIZERS.build(pose_estimator.cfg.visualizer)
# the dataset_meta is loaded from the checkpoint and
# then pass to the model in init_pose_estimator
visualizer.set_dataset_meta(
    pose_estimator.dataset_meta, skeleton_style='mmpose')

Loads checkpoint by local backend from path: checkpoints/hands/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth


In [6]:
with open(input_file, 'rb') as f:
    bboxes = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'bboxes/tracked_bboxes_gopro1_synced.pkl'

In [17]:
for k,v in bboxes.items():
    for obj_id in range(2):
        if obj_id not in v.keys() or len(v[obj_id]) == 0:
            bboxes[k][obj_id] = np.array([[0, 0, 0, 0]])

In [18]:
bboxes

{0: {0: array([[913, 688, 964, 740]], dtype=int64),
  1: array([[ 974,  426, 1046,  465]], dtype=int64)},
 1: {0: array([[913, 689, 963, 741]], dtype=int64),
  1: array([[ 975,  426, 1046,  466]], dtype=int64)},
 2: {0: array([[913, 689, 963, 741]], dtype=int64),
  1: array([[ 977,  426, 1045,  465]], dtype=int64)},
 3: {0: array([[913, 688, 963, 741]], dtype=int64),
  1: array([[ 975,  427, 1045,  466]], dtype=int64)},
 4: {0: array([[913, 688, 963, 741]], dtype=int64),
  1: array([[ 975,  427, 1045,  466]], dtype=int64)},
 5: {0: array([[913, 688, 963, 741]], dtype=int64),
  1: array([[ 975,  426, 1046,  466]], dtype=int64)},
 6: {0: array([[913, 688, 962, 741]], dtype=int64),
  1: array([[ 976,  426, 1046,  466]], dtype=int64)},
 7: {0: array([[913, 688, 962, 741]], dtype=int64),
  1: array([[ 976,  425, 1046,  466]], dtype=int64)},
 8: {0: array([[913, 688, 963, 741]], dtype=int64),
  1: array([[ 976,  423, 1046,  466]], dtype=int64)},
 9: {0: array([[913, 688, 962, 741]], dtype=in

In [19]:


resolution = (1920, 1080)

frame_idx = 0 # start frame

# Load the given frame
cap = cv2.VideoCapture(video_file)  # Replace with your image path
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video file: {video_file}")

video_writer = None
pred_instances_list = []
frame_idx = 0

cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

while cap.isOpened():
    success, frame = cap.read()
    

    if not success:
        break

    print(bboxes[frame_idx])
    print(bboxes[frame_idx].values())
    bbox = np.array(list(bboxes[frame_idx].values()))
    bbox_shape = bbox.shape
    print(bbox_shape)
    
    if bbox_shape[-1] == 4:

        # topdown pose estimation
        pred_instances = process_one_image(frame, bbox.reshape(bbox_shape[0]*bbox_shape[1], bbox_shape[2]), pose_estimator, visualizer, 0.001)

        # save prediction results
        pred_instances_list.append(
            dict(
                frame_id=frame_idx,
                instances=split_instances(pred_instances)))

        # output videos
        frame_vis = visualizer.get_image()

        if video_writer is None:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            # the size of the image with visualization may vary
            # depending on the presence of heatmaps
            video_writer = cv2.VideoWriter(
                output_file,
                fourcc,
                30,  # saved fps
                (frame_vis.shape[1], frame_vis.shape[0]))

        video_writer.write(mmcv.rgb2bgr(frame_vis))

        if show:
            # press ESC to exit
            if cv2.waitKey(5) & 0xFF == 27:
                break

            time.sleep(0)

    frame_idx += 1

if video_writer:
    video_writer.release()

cap.release()

with open(save_path, 'w') as f:
    json.dump(
        dict(
            meta_info=pose_estimator.dataset_meta,
            instance_info=pred_instances_list),
        f,
        indent='\t')
print(f'predictions have been saved at {save_path}')


{0: array([[913, 688, 964, 740]], dtype=int64), 1: array([[ 974,  426, 1046,  465]], dtype=int64)}
dict_values([array([[913, 688, 964, 740]], dtype=int64), array([[ 974,  426, 1046,  465]], dtype=int64)])
(2, 1, 4)


c:\Users\fisv\AppData\Local\miniconda3\envs\openpose\Lib\site-packages\mmdet\models\layers\se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
c:\Users\fisv\AppData\Local\miniconda3\envs\openpose\Lib\site-packages\mmdet\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


{0: array([[913, 689, 963, 741]], dtype=int64), 1: array([[ 975,  426, 1046,  466]], dtype=int64)}
dict_values([array([[913, 689, 963, 741]], dtype=int64), array([[ 975,  426, 1046,  466]], dtype=int64)])
(2, 1, 4)
{0: array([[913, 689, 963, 741]], dtype=int64), 1: array([[ 977,  426, 1045,  465]], dtype=int64)}
dict_values([array([[913, 689, 963, 741]], dtype=int64), array([[ 977,  426, 1045,  465]], dtype=int64)])
(2, 1, 4)
{0: array([[913, 688, 963, 741]], dtype=int64), 1: array([[ 975,  427, 1045,  466]], dtype=int64)}
dict_values([array([[913, 688, 963, 741]], dtype=int64), array([[ 975,  427, 1045,  466]], dtype=int64)])
(2, 1, 4)
{0: array([[913, 688, 963, 741]], dtype=int64), 1: array([[ 975,  427, 1045,  466]], dtype=int64)}
dict_values([array([[913, 688, 963, 741]], dtype=int64), array([[ 975,  427, 1045,  466]], dtype=int64)])
(2, 1, 4)
{0: array([[913, 688, 963, 741]], dtype=int64), 1: array([[ 975,  426, 1046,  466]], dtype=int64)}
dict_values([array([[913, 688, 963, 741]]